In [62]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import datetime
from datetime import date

from dataclasses import dataclass
from typing import Literal
import logfire
from pydantic import BaseModel, Field
from rich.prompt import Prompt

from pydantic_ai import (
    Agent,
    ModelMessage,
    ModelRetry,
    RunContext,
    RunUsage,
    UsageLimits,
    WebFetchTool,
)

In [50]:
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-eu.pydantic.dev/donagheyruairi/dublin-events

In [57]:
class EventDetails(BaseModel):
    """Details of an event on in Dublin."""

    name: str
    startTime: date
    endTime: date

class NoEvent(BaseModel):
    """No event exists."""

In [ ]:
@dataclass
class Deps:
    web_page_text: str
    req_date: date

search_agent = Agent[Deps, EventDetails | NoEvent](
    'openai:gpt-5.4-nano',
    output_type= EventDetails | NoEvent,  # type: ignore
    retries=3,
    system_prompt=(
        'Your job is to find events in Dublin on the given date.'
    ),
)

@search_agent.tool
async def extract_event(ctx: RunContext[Deps]) -> list[EventDetails]:
    """Get details of events."""
    result = await extraction_agent.run(ctx.deps.web_page_text, usage=ctx.usage)
    logfire.info('found {event_count} events', event_count=len(result.output))
    return result.output

extraction_agent = Agent(
    'openai:gpt-5.4-nano',
    builtin_tools=[WebFetchTool()],
    output_type=list[EventDetails],
    system_prompt='Extract all the event details from the given text.',
)

In [59]:
text = """

Who: The Climate Heist
Where: The Sugar Club
When: Tue 14th Apr

Who: Niamh McCrystal
Where: Whelan's
When: Tue 14th Apr

Who: David Keenan
Where: Whelan's
When: Tue 14th Apr - Wed 15th Apr
"""

usage_limits = UsageLimits(request_limit=10)



In [60]:
async def main():
    deps = Deps(
        web_page_text=text,
        req_date=datetime.date(2025, 4, 15),
    )
    message_history: list[ModelMessage] | None = None
    usage: RunUsage = RunUsage()
    while True:
        result = await search_agent.run(
            f'Find me an event from on {deps.req_date}',
            deps=deps,
            usage=usage,
            message_history=message_history,
            usage_limits=usage_limits,
        )
        if isinstance(result.output, NoEvent):
            print('Nothing found.')
            break
        else:
            event = result.output
            print(f'Event found: {event}')
            break

In [61]:
import asyncio
await main()

16:52:53.500 search_agent run
16:52:53.508   chat gpt-5.4-nano
16:52:54.979   running tool: extract_event
16:52:54.983     extraction_agent run
16:52:54.986       chat gpt-5.4-nano
16:52:56.741     found 3 events
16:52:56.749   chat gpt-5.4-nano
Event found: name='David Keenan' startTime=datetime.date(2026, 4, 14) endTime=datetime.date(2026, 4, 15)
